# Phase 6 — Sequential Learner Modelling

This notebook tests whether a compact LSTM-based Deep Knowledge Tracing (DKT) model adds useful evidence beyond the existing first-order transition baseline and the selected gated hybrid recommender.

The experiment has two deliberately separate outcomes:

1. **Prediction:** can the recurrent model predict correctness on a learner's next supported skill interaction?
2. **Recommendation:** does its learner state improve top-$K$ problem recommendations when it is added as a bounded reranker or fallback?



## Step 1 — Mount Google Drive


In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


## Step 2 — Configure the experiment



In [2]:
from pathlib import Path
import gc
import json
import math
import time

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from sklearn.metrics import roc_auc_score, log_loss, brier_score_loss
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
tf.keras.utils.set_random_seed(RANDOM_STATE)

DRIVE_ROOT = Path('/content/drive/MyDrive/datasets')
RAW_PATH = DRIVE_ROOT / '2012-2013-data-with-predictions-4-final.csv'
PHASE2_ROOT = DRIVE_ROOT / 'recommendation_benchmark_final_outputs' / 'assistments'
PHASE3_ROOT = DRIVE_ROOT / 'recommendation_baseline_outputs'
PHASE4_TUNING_ROOT = DRIVE_ROOT / 'hybrid_recommender_gated_outputs'
PHASE4_FINAL_ROOT = DRIVE_ROOT / 'hybrid_recommender_gated_final_outputs'
OUTPUT_ROOT = DRIVE_ROOT / 'sequential_learner_model_outputs'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

MAX_SEQUENCE_LENGTH = 200
MAX_FUTURE_EVENTS_PER_LEARNER = 100
BATCH_SIZE = 128
EMBEDDING_DIM = 48
HIDDEN_DIM = 64
MAX_EPOCHS = 12
EARLY_STOPPING_PATIENCE = 2
MASTERY_PRIOR_STRENGTH = 5.0
TOP_SKILLS_PER_LEARNER = 5
PROBLEMS_PER_SKILL = 20
CANDIDATE_DEPTH = 100
FINAL_K = 20
RRF_CONSTANT = 60.0
TARGET_SUCCESS_BANDS = (0.55, 0.65, 0.75)
DKT_WEIGHTS = (0.0, 0.02, 0.05, 0.10, 0.25)
METRIC_KS = (5, 10, 20)
BOOTSTRAP_RESAMPLES = 2000

required_paths = [
    RAW_PATH,
    PHASE2_ROOT / 'learner_splits.parquet',
    PHASE2_ROOT / 'skill_catalog.parquet',
    PHASE2_ROOT / 'problem_catalog.parquet',
    PHASE2_ROOT / 'problem_skill_map.parquet',
    PHASE2_ROOT / 'skill_name_id_map.parquet',
    PHASE2_ROOT / 'early_problem_history.parquet',
    PHASE2_ROOT / 'future_problem_relevance.parquet',
    PHASE3_ROOT / 'recommendation_metrics.csv',
    PHASE4_TUNING_ROOT / 'gated_experiment_split.csv',
    PHASE4_TUNING_ROOT / 'gated_tuning_components.parquet',
    PHASE4_FINAL_ROOT / 'hybrid_recommendations.parquet',
]

print('TensorFlow:', tf.__version__)
print('Output directory:', OUTPUT_ROOT)


TensorFlow: 2.20.0
Output directory: /content/drive/MyDrive/datasets/sequential_learner_model_outputs


## Step 3 — Load the frozen handoff



In [3]:
learners = pd.read_parquet(PHASE2_ROOT / 'learner_splits.parquet')
skill_catalog = pd.read_parquet(PHASE2_ROOT / 'skill_catalog.parquet')
problem_catalog = pd.read_parquet(PHASE2_ROOT / 'problem_catalog.parquet')
problem_skill_map = pd.read_parquet(PHASE2_ROOT / 'problem_skill_map.parquet')
skill_name_map = pd.read_parquet(PHASE2_ROOT / 'skill_name_id_map.parquet')
early_problem = pd.read_parquet(PHASE2_ROOT / 'early_problem_history.parquet')
future_problem = pd.read_parquet(PHASE2_ROOT / 'future_problem_relevance.parquet')
phase3_metrics = pd.read_csv(PHASE3_ROOT / 'recommendation_metrics.csv')

for frame in [learners, early_problem, future_problem]:
    if 'learner_id' in frame:
        frame['learner_id'] = frame['learner_id'].astype(str)
for frame in [skill_catalog, problem_catalog, early_problem, future_problem]:
    if 'item_id' in frame:
        frame['item_id'] = frame['item_id'].astype(str)
problem_skill_map['problem_item_id'] = problem_skill_map['problem_item_id'].astype(str)
problem_skill_map['skill_item_id'] = problem_skill_map['skill_item_id'].astype(str)

split = pd.read_csv(
    PHASE4_TUNING_ROOT / 'gated_experiment_split.csv',
    dtype={'learner_id': str},
)
if {'Dataset', 'Task'}.issubset(split.columns):
    split = split[
        split['Dataset'].eq('ASSISTments') & split['Task'].eq('problem')
    ].copy()
role_column = 'DiscoveryRole' if 'DiscoveryRole' in split.columns else 'Role'
training_roles = {'training', 'discovery_training'}
tuning_roles = {'tuning', 'discovery_tuning'}
discovery_train_ids = sorted(split.loc[
    split[role_column].isin(training_roles), 'learner_id'
].astype(str).unique())
discovery_tuning_ids = sorted(split.loc[
    split[role_column].isin(tuning_roles), 'learner_id'
].astype(str).unique())
validation_ids = sorted(learners.loc[
    learners['cohort'].eq('validation'), 'learner_id'
].astype(str).unique())
expected_discovery_ids = set(learners.loc[
    learners['cohort'].eq('discovery'), 'learner_id'
].astype(str))


skill_items = sorted(skill_catalog['item_id'].astype(str).unique())
skill_to_index = {item: index for index, item in enumerate(skill_items)}
index_to_skill = {index: item for item, index in skill_to_index.items()}
supported_skills = set(skill_items)

cohort_summary = pd.DataFrame([
    {'Role': 'discovery_training', 'Learners': len(discovery_train_ids)},
    {'Role': 'discovery_tuning', 'Learners': len(discovery_tuning_ids)},
    {'Role': 'validation', 'Learners': len(validation_ids)},
    {'Role': 'supported_skill_vocabulary', 'Learners': len(skill_items)},
])
display(cohort_summary)


,Role,Learners
0,discovery_training,21334
1,discovery_tuning,5334
2,validation,6667
3,supported_skill_vocabulary,161


## Step 4 — Reconstruct chronological skill-response sequences

The temporal benchmark stores aggregate histories, so this stage reads the ASSISTments interactions to recover event order. Rows are cleaned with the same rules used previously. The number of early events saved for each learner determines the boundary between early and future interactions.

Skill identity follows the frozen contract: a native skill ID is preferred, a discovery-early name-to-ID mapping is used when the native ID is absent, and a normalised text fallback is used only when it already belongs to the frozen catalogue. Unsupported skills are removed after the chronological boundary has been assigned, preventing the model vocabulary from learning from future data.


In [4]:
def normalise_learner_ids(values):
    return (
        values.astype(str).str.strip()
        .str.replace(r'^(-?\d+)\.0$', r'\1', regex=True)
    )


def normalise_skill_names(values):
    return (
        values.fillna('').astype(str).str.strip().str.casefold()
        .str.replace(r'\s+', ' ', regex=True)
    )


raw_columns = [
    'problem_log_id', 'user_id', 'skill', 'skill_id',
    'start_time', 'end_time', 'correct',
]
eligible_ids = set(learners['learner_id'])
raw_parts = []
for chunk in pd.read_csv(
    RAW_PATH, usecols=raw_columns, chunksize=750_000, low_memory=False,
):
    chunk['learner_id'] = normalise_learner_ids(chunk['user_id'])
    chunk['correct'] = pd.to_numeric(chunk['correct'], errors='coerce')
    chunk = chunk[
        chunk['learner_id'].isin(eligible_ids)
        & chunk['correct'].isin([0, 1])
    ].copy()
    if not chunk.empty:
        raw_parts.append(chunk)

events = pd.concat(raw_parts, ignore_index=True)
del raw_parts
events['problem_log_id'] = pd.to_numeric(events['problem_log_id'], errors='coerce')
duplicate_event = (
    events['problem_log_id'].notna()
    & events.duplicated('problem_log_id', keep='first')
)
events = events.loc[~duplicate_event].copy()
events['start_time'] = pd.to_datetime(events['start_time'], errors='coerce')
events['end_time'] = pd.to_datetime(events['end_time'], errors='coerce')
events['event_time'] = events['start_time'].fillna(events['end_time'])
events['normalised_skill_name'] = normalise_skill_names(events['skill'])

name_lookup = skill_name_map.set_index('normalised_skill_name')['mapped_skill_id']
native = pd.to_numeric(events['skill_id'], errors='coerce').astype('Float64')
mapped = events['normalised_skill_name'].map(name_lookup).astype('Float64')
resolved = native.fillna(mapped)
skill_item = pd.Series(pd.NA, index=events.index, dtype='object')
has_id = resolved.notna()
skill_item.loc[has_id] = (
    'skill:id:' + resolved.loc[has_id].astype('Int64').astype(str)
)
valid_text = (
    resolved.isna()
    & ~events['normalised_skill_name'].isin({'', 'unknown', 'nan', 'none'})
)
skill_item.loc[valid_text] = (
    'skill:text:' + events.loc[valid_text, 'normalised_skill_name']
)
events['skill_item'] = skill_item

events = events.sort_values(
    ['learner_id', 'event_time', 'problem_log_id'],
    na_position='last', kind='mergesort',
).copy()
events['event_number'] = events.groupby('learner_id').cumcount()
cutoffs = learners.set_index('learner_id')['early_interactions'].astype(int)
events['early_cutoff'] = events['learner_id'].map(cutoffs)
events['is_early'] = events['event_number'] < events['early_cutoff']
events['cohort'] = events['learner_id'].map(
    learners.set_index('learner_id')['cohort']
)

raw_event_counts = events.groupby('learner_id').size()
expected_event_counts = learners.set_index('learner_id')[
    ['early_interactions', 'future_interactions']
].sum(axis=1)
count_match = raw_event_counts.reindex(expected_event_counts.index).fillna(0).astype(int).eq(
    expected_event_counts.astype(int)
)

if not count_match.all():
    mismatched = count_match[~count_match].index[:10].tolist()
    raise ValueError(
        'Raw reconstruction does not match the frozen Phase 2 event counts. '
        f'Example learner IDs: {mismatched}'
    )

sequence_events = events[
    events['skill_item'].isin(supported_skills)
][['learner_id', 'problem_log_id', 'event_number', 'is_early',
   'cohort', 'skill_item', 'correct']].copy()
sequence_events['skill_index'] = sequence_events['skill_item'].map(skill_to_index).astype(int)
sequence_events['correct'] = sequence_events['correct'].astype('float32')

sequence_diagnostics = pd.DataFrame([{
    'RawEligibleEvents': len(events),
    'SupportedSkillEvents': len(sequence_events),
    'SupportedEventRate': len(sequence_events) / len(events),
    'LearnersWithExactFrozenEventCount': int(count_match.sum()),
    'EligibleLearners': len(count_match),
    'ExactFrozenEventCountRate': float(count_match.mean()),
    'DiscoveryEarlySupportedEvents': int((
        sequence_events['cohort'].eq('discovery') & sequence_events['is_early']
    ).sum()),
    'ValidationEarlySupportedEvents': int((
        sequence_events['cohort'].eq('validation') & sequence_events['is_early']
    ).sum()),
    'MaximumSequenceLength': MAX_SEQUENCE_LENGTH,
}])
display(sequence_diagnostics)
del events, raw_event_counts, expected_event_counts, count_match
gc.collect()


,RawEligibleEvents,SupportedSkillEvents,SupportedEventRate,LearnersWithExactFrozenEventCount,EligibleLearners,ExactFrozenEventCountRate,DiscoveryEarlySupportedEvents,ValidationEarlySupportedEvents,MaximumSequenceLength
0,5999086,2622512,0.437152,33335,33335,1.0,1448381,360780,200


116

## Step 5 — Build training examples and the LSTM-DKT model

Each input token combines the current skill with whether the learner answered it correctly. The LSTM processes these ordered tokens, and its output at each time step contains a predicted correctness probability for every supported skill. The loss selects only the probability corresponding to the actual next skill.

In [5]:
def encode_interaction(skill_indices, correctness):
    return (
        1 + skill_indices.astype('int32')
        + correctness.astype('int32') * len(skill_items)
    )


def build_training_examples(frame, learner_ids):
    learner_set = set(map(str, learner_ids))
    source = frame[
        frame['learner_id'].isin(learner_set) & frame['is_early']
    ].sort_values(['learner_id', 'event_number'], kind='mergesort')
    examples = []
    for learner_id, group in source.groupby('learner_id', sort=False):
        skills = group['skill_index'].to_numpy(dtype='int32')
        outcomes = group['correct'].to_numpy(dtype='float32')
        if len(skills) < 2:
            continue
        skills = skills[-(MAX_SEQUENCE_LENGTH + 1):]
        outcomes = outcomes[-(MAX_SEQUENCE_LENGTH + 1):]
        tokens = encode_interaction(skills[:-1], outcomes[:-1])
        targets = np.column_stack([skills[1:], outcomes[1:]]).astype('float32')
        weights = np.ones(len(tokens), dtype='float32')
        examples.append((tokens, targets, weights))
    return examples


def dataset_from_examples(examples, shuffle):
    def generator():
        order = np.arange(len(examples))
        if shuffle:
            np.random.default_rng(RANDOM_STATE).shuffle(order)
        for index in order:
            yield examples[index]

    signature = (
        tf.TensorSpec(shape=(None,), dtype=tf.int32),
        tf.TensorSpec(shape=(None, 2), dtype=tf.float32),
        tf.TensorSpec(shape=(None,), dtype=tf.float32),
    )
    dataset = tf.data.Dataset.from_generator(generator, output_signature=signature)
    return dataset.padded_batch(
        BATCH_SIZE,
        padded_shapes=([None], [None, 2], [None]),
        padding_values=(np.int32(0), np.float32(0), np.float32(0)),
    ).prefetch(tf.data.AUTOTUNE)


@keras.utils.register_keras_serializable(package='Phase6')
def skill_aware_binary_crossentropy(y_true, y_pred):
    target_skill = tf.cast(y_true[..., 0], tf.int32)
    target_outcome = y_true[..., 1]
    selected_probability = tf.gather(
        y_pred, target_skill, axis=2, batch_dims=2
    )
    return keras.backend.binary_crossentropy(target_outcome, selected_probability)


training_examples = build_training_examples(sequence_events, discovery_train_ids)
tuning_examples = build_training_examples(sequence_events, discovery_tuning_ids)
training_dataset = dataset_from_examples(training_examples, shuffle=True)
tuning_dataset = dataset_from_examples(tuning_examples, shuffle=False)

interaction_input = keras.Input(shape=(None,), dtype='int32', name='interaction_token')
embedded = layers.Embedding(
    input_dim=2 * len(skill_items) + 1,
    output_dim=EMBEDDING_DIM,
    mask_zero=True,
    name='interaction_embedding',
)(interaction_input)
hidden = layers.LSTM(
    HIDDEN_DIM, return_sequences=True, dropout=0.15, name='learner_state_lstm'
)(embedded)
skill_probabilities = layers.Dense(
    len(skill_items), activation='sigmoid', name='skill_correctness'
)(hidden)
dkt_model = keras.Model(interaction_input, skill_probabilities, name='compact_lstm_dkt')
dkt_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss=skill_aware_binary_crossentropy,
)
dkt_model.summary()
print('Training learners with usable sequences:', len(training_examples))
print('Tuning learners with usable sequences:', len(tuning_examples))


Model: "compact_lstm_dkt"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ interaction_token   │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ interaction_embedd… │ (None, None, 48)  │     15,504 │ interaction_toke… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, None)      │          0 │ interaction_toke… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ learner_state_lstm  │ (None, None, 64)  │     28,928 │ interaction_embe… │
│ (LSTM)              │                   │            │ not_equal[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ skill_correctness   │ (None, None, 161) │     10,465 │ learner_state_ls… │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 54,897 (214.44 KB)

 Trainable params: 54,897 (214.44 KB)

 Non-trainable params: 0 (0.00 B)

Training learners with usable sequences: 13010
Tuning learners with usable sequences: 3243


## Step 6 — Train using discovery-early interactions

Training stops when the loss on discovery-tuning early sequences no longer improves. The best discovery checkpoint is restored before any future-period evaluation.

Reducing `MAX_SEQUENCE_LENGTH`, `HIDDEN_DIM`, or `MAX_EPOCHS` for less computation

In [6]:
checkpoint_path = OUTPUT_ROOT / 'best_discovery_dkt.weights.h5'
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=EARLY_STOPPING_PATIENCE,
        restore_best_weights=True,
    ),
    keras.callbacks.ModelCheckpoint(
        checkpoint_path, monitor='val_loss', save_best_only=True,
        save_weights_only=True,
    ),
]
training_started = time.perf_counter()
history = dkt_model.fit(
    training_dataset,
    validation_data=tuning_dataset,
    epochs=MAX_EPOCHS,
    callbacks=callbacks,
    verbose=1,
)
training_seconds = time.perf_counter() - training_started
training_history = pd.DataFrame(history.history)
training_history.insert(0, 'Epoch', np.arange(1, len(training_history) + 1))
training_history['TrainingSecondsTotal'] = training_seconds
display(training_history)


Epoch 1/12
    102/Unknown 51s 464ms/step - loss: 0.6408

/usr/local/lib/python3.13/dist-packages/keras/src/trainers/epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


102/102 ━━━━━━━━━━━━━━━━━━━━ 55s 510ms/step - loss: 0.6055 - val_loss: 0.5662
Epoch 2/12
102/102 ━━━━━━━━━━━━━━━━━━━━ 53s 516ms/step - loss: 0.5590 - val_loss: 0.5547
Epoch 3/12
102/102 ━━━━━━━━━━━━━━━━━━━━ 53s 522ms/step - loss: 0.5514 - val_loss: 0.5500
Epoch 4/12
102/102 ━━━━━━━━━━━━━━━━━━━━ 51s 499ms/step - loss: 0.5474 - val_loss: 0.5471
Epoch 5/12
102/102 ━━━━━━━━━━━━━━━━━━━━ 54s 528ms/step - loss: 0.5450 - val_loss: 0.5453
Epoch 6/12
102/102 ━━━━━━━━━━━━━━━━━━━━ 82s 804ms/step - loss: 0.5431 - val_loss: 0.5443
Epoch 7/12
102/102 ━━━━━━━━━━━━━━━━━━━━ 53s 517ms/step - loss: 0.5416 - val_loss: 0.5434
Epoch 8/12
102/102 ━━━━━━━━━━━━━━━━━━━━ 83s 524ms/step - loss: 0.5405 - val_loss: 0.5428
Epoch 9/12
102/102 ━━━━━━━━━━━━━━━━━━━━ 82s 806ms/step - loss: 0.5394 - val_loss: 0.5422
Epoch 10/12
102/102 ━━━━━━━━━━━━━━━━━━━━ 51s 500ms/step - loss: 0.5385 - val_loss: 0.5418
Epoch 11/12
102/102 ━━━━━━━━━━━━━━━━━━━━ 52s 509ms/step - loss: 0.5376 - val_loss: 0.5414
Epoch 12/12
102/102 ━━━━━━━━━━

,Epoch,loss,val_loss,TrainingSecondsTotal
0,1,0.605548,0.566242,721.165392
1,2,0.558966,0.554711,721.165392
2,3,0.551434,0.549963,721.165392
3,4,0.547441,0.547135,721.165392
4,5,0.545018,0.545264,721.165392
5,6,0.543084,0.544301,721.165392
6,7,0.541643,0.543438,721.165392
7,8,0.540487,0.542793,721.165392
8,9,0.539394,0.542189,721.165392
9,10,0.538484,0.541767,721.165392


## Step 7 — Evaluate correctness prediction and calibration

The model is first evaluated on discovery-tuning future interactions. For each learner, the final part of the early history supplies context and future responses are revealed in chronological order for teacher-forced next-response evaluation. A shrunk per-skill success-rate model fitted on discovery-training early data provides a simple non-recurrent comparator. Validation-future prediction is deliberately deferred until after the recommendation-integration settings are frozen.

ROC-AUC measures discrimination, while log loss and Brier score measure probability quality. Calibration tables show whether predicted probabilities correspond to observed success rates. These results evaluate knowledge-state prediction.


In [7]:
training_early = sequence_events[
    sequence_events['learner_id'].isin(set(discovery_train_ids))
    & sequence_events['is_early']
]
global_success = float(training_early['correct'].mean())
skill_prior_table = training_early.groupby('skill_index').agg(
    Correct=('correct', 'sum'), Interactions=('correct', 'size')
).reindex(range(len(skill_items)), fill_value=0)
skill_prior = (
    skill_prior_table['Correct'].to_numpy()
    + MASTERY_PRIOR_STRENGTH * global_success
) / (skill_prior_table['Interactions'].to_numpy() + MASTERY_PRIOR_STRENGTH)


def future_prediction_rows(frame, learner_ids, cohort_name):
    rows = []
    learner_set = set(map(str, learner_ids))
    source = frame[frame['learner_id'].isin(learner_set)].sort_values(
        ['learner_id', 'event_number'], kind='mergesort'
    )
    examples = []
    metadata = []
    for learner_id, group in source.groupby('learner_id', sort=False):
        early = group[group['is_early']].tail(MAX_SEQUENCE_LENGTH)
        future = group[~group['is_early']].head(MAX_FUTURE_EVENTS_PER_LEARNER)
        combined = pd.concat([early, future], ignore_index=True)
        if len(combined) < 2 or future.empty:
            continue
        skills = combined['skill_index'].to_numpy(dtype='int32')
        outcomes = combined['correct'].to_numpy(dtype='float32')
        future_mask = (~combined['is_early']).to_numpy()[1:]
        tokens = encode_interaction(skills[:-1], outcomes[:-1])
        examples.append((tokens, skills[1:], outcomes[1:], future_mask))
        metadata.append(learner_id)

    for start in range(0, len(examples), BATCH_SIZE):
        batch = examples[start:start + BATCH_SIZE]
        max_length = max(len(value[0]) for value in batch)
        token_batch = np.zeros((len(batch), max_length), dtype='int32')
        for row_index, (tokens, _, _, _) in enumerate(batch):
            token_batch[row_index, :len(tokens)] = tokens
        predictions = dkt_model.predict(token_batch, verbose=0)
        for row_index, (_, target_skills, outcomes, mask) in enumerate(batch):
            positions = np.flatnonzero(mask)
            selected = predictions[row_index, positions, target_skills[positions]]
            for position, probability in zip(positions, selected):
                skill_index = int(target_skills[position])
                rows.append({
                    'Cohort': cohort_name,
                    'learner_id': metadata[start + row_index],
                    'skill_item_id': index_to_skill[skill_index],
                    'ActualCorrect': float(outcomes[position]),
                    'DKTProbability': float(probability),
                    'EmpiricalSkillProbability': float(skill_prior[skill_index]),
                })
        del predictions, token_batch
    return pd.DataFrame(rows)


tuning_predictions = future_prediction_rows(
    sequence_events, discovery_tuning_ids, 'discovery_tuning_future'
)

def summarise_predictions(prediction_rows):
    prediction_metric_rows = []
    calibration_parts = []
    for cohort_name, group in prediction_rows.groupby('Cohort', sort=False):
        for model_name, probability_column in {
            'lstm_dkt': 'DKTProbability',
            'shrunk_skill_success': 'EmpiricalSkillProbability',
        }.items():
            actual = group['ActualCorrect'].to_numpy()
            probability = group[probability_column].clip(1e-6, 1 - 1e-6).to_numpy()
            prediction_metric_rows.append({
                'Cohort': cohort_name,
                'Model': model_name,
                'Interactions': len(group),
                'Learners': group['learner_id'].nunique(),
                'ROCAUC': roc_auc_score(actual, probability),
                'LogLoss': log_loss(actual, probability, labels=[0, 1]),
                'BrierScore': brier_score_loss(actual, probability),
            })
            calibrated = pd.DataFrame({'Actual': actual, 'Probability': probability})
            calibrated['ProbabilityBand'] = pd.cut(
                calibrated['Probability'], bins=np.linspace(0, 1, 11),
                include_lowest=True,
            ).astype(str)
            summary = calibrated.groupby('ProbabilityBand', observed=False).agg(
                Predictions=('Actual', 'size'),
                MeanPredicted=('Probability', 'mean'),
                ObservedSuccess=('Actual', 'mean'),
            ).reset_index()
            summary.insert(0, 'Model', model_name)
            summary.insert(0, 'Cohort', cohort_name)
            calibration_parts.append(summary)
    return pd.DataFrame(prediction_metric_rows), pd.concat(calibration_parts, ignore_index=True)

prediction_metrics, calibration = summarise_predictions(tuning_predictions)
display(prediction_metrics)
display(calibration[calibration['Predictions'].gt(0)].head(20))


,Cohort,Model,Interactions,Learners,ROCAUC,LogLoss,BrierScore
0,discovery_tuning_future,lstm_dkt,106023,3106,0.720641,0.546765,0.183176
1,discovery_tuning_future,shrunk_skill_success,106023,3106,0.601101,0.602491,0.206743


,Cohort,Model,ProbabilityBand,Predictions,MeanPredicted,ObservedSuccess
0,discovery_tuning_future,lstm_dkt,"(-0.001, 0.1]",146,0.065503,0.095890
1,discovery_tuning_future,lstm_dkt,"(0.1, 0.2]",996,0.160722,0.152610
2,discovery_tuning_future,lstm_dkt,"(0.2, 0.3]",2439,0.256049,0.252563
3,discovery_tuning_future,lstm_dkt,"(0.3, 0.4]",4109,0.355383,0.338038
4,discovery_tuning_future,lstm_dkt,"(0.4, 0.5]",7129,0.453916,0.462056
5,discovery_tuning_future,lstm_dkt,"(0.5, 0.6]",11692,0.553809,0.556107
6,discovery_tuning_future,lstm_dkt,"(0.6, 0.7]",18093,0.653419,0.649588
7,discovery_tuning_future,lstm_dkt,"(0.7, 0.8]",26756,0.752712,0.750075
8,discovery_tuning_future,lstm_dkt,"(0.8, 0.9]",26383,0.846534,0.839935
9,discovery_tuning_future,lstm_dkt,"(0.9, 1.0]",8280,0.930374,0.915821


## Step 8 — Extract the current learner state

For recommendation, the model uses only each learner's early supported history. The final recurrent output gives a predicted correctness probability for every supported skill. Learners without usable supported history receive the shrunk discovery prior and are explicitly labelled as prior fallbacks.

The saved state includes an evidence-strength field based on early supported interaction count. This field records how much history informed the state.


In [8]:
def extract_skill_states(frame, learner_ids):
    learner_ids = list(map(str, learner_ids))
    early = frame[
        frame['learner_id'].isin(set(learner_ids)) & frame['is_early']
    ].sort_values(['learner_id', 'event_number'], kind='mergesort')
    grouped = {learner: group for learner, group in early.groupby('learner_id', sort=False)}
    state_matrix = np.tile(skill_prior.astype('float32'), (len(learner_ids), 1))
    evidence_counts = np.zeros(len(learner_ids), dtype='int32')
    state_sources = np.full(len(learner_ids), 'discovery_skill_prior', dtype=object)

    for start in range(0, len(learner_ids), BATCH_SIZE):
        batch_ids = learner_ids[start:start + BATCH_SIZE]
        sequences = []
        active_rows = []
        for local_index, learner_id in enumerate(batch_ids):
            group = grouped.get(learner_id)
            if group is None or group.empty:
                continue
            group = group.tail(MAX_SEQUENCE_LENGTH)
            skills = group['skill_index'].to_numpy(dtype='int32')
            outcomes = group['correct'].to_numpy(dtype='float32')
            sequences.append(encode_interaction(skills, outcomes))
            active_rows.append((local_index, len(skills)))
        if not sequences:
            continue
        max_length = max(map(len, sequences))
        token_batch = np.zeros((len(sequences), max_length), dtype='int32')
        for row_index, tokens in enumerate(sequences):
            token_batch[row_index, :len(tokens)] = tokens
        predictions = dkt_model.predict(token_batch, verbose=0)
        for prediction_row, (local_index, length) in enumerate(active_rows):
            absolute_index = start + local_index
            state_matrix[absolute_index] = predictions[prediction_row, length - 1]
            evidence_counts[absolute_index] = length
            state_sources[absolute_index] = 'lstm_dkt_early_history'
        del predictions, token_batch

    summary = pd.DataFrame({
        'learner_id': learner_ids,
        'SupportedEarlyInteractionsUsed': evidence_counts,
        'StateSource': state_sources,
        'HistoryEvidenceStrength': evidence_counts / (evidence_counts + 20.0),
    })
    return state_matrix, summary


tuning_state_matrix, tuning_state_summary = extract_skill_states(
    sequence_events, discovery_tuning_ids
)
validation_state_matrix, validation_state_summary = extract_skill_states(
    sequence_events, validation_ids
)

validation_skill_states = pd.DataFrame({
    'learner_id': np.repeat(validation_ids, len(skill_items)),
    'skill_item_id': np.tile(skill_items, len(validation_ids)),
    'PredictedCorrectness': validation_state_matrix.reshape(-1),
})
validation_skill_states = validation_skill_states.merge(
    validation_state_summary, on='learner_id', how='left', validate='many_to_one'
)
display(validation_state_summary['StateSource'].value_counts(dropna=False))
display(validation_skill_states.head())


,count
StateSource,
lstm_dkt_early_history,4162
discovery_skill_prior,2505


,learner_id,skill_item_id,PredictedCorrectness,SupportedEarlyInteractionsUsed,StateSource,HistoryEvidenceStrength
0,100516,skill:id:1,0.554047,44,lstm_dkt_early_history,0.6875
1,100516,skill:id:10,0.891977,44,lstm_dkt_early_history,0.6875
2,100516,skill:id:101,0.637355,44,lstm_dkt_early_history,0.6875
3,100516,skill:id:103,0.892785,44,lstm_dkt_early_history,0.6875
4,100516,skill:id:104,0.733566,44,lstm_dkt_early_history,0.6875


## Step 9 — Build sparse DKT problem candidates and the existing tuning hybrid

Predicted skill correctness is converted into a bounded suitability signal. Skills near a target success probability are preferred: this avoids recommending only already-mastered material or automatically selecting the hardest material. For each selected skill, the notebook retrieves a small number of supported problems through the frozen discovery-early problem-to-skill map.

The discovery version of the current gated hybrid is reconstructed from the Phase 4 tuning components using its selected structure: collaborative filtering is primary, content contributes a 0.02 reciprocal-rank bonus, sequence is used when collaborative evidence is absent, and popularity is the final fallback. This produces a fair discovery-only base for DKT integration tuning without touching validation-future labels.


In [9]:
problem_popularity = problem_catalog.set_index('item_id')['training_interactions'].astype(float)
problem_popularity = problem_popularity / problem_popularity.max()
problem_map = problem_skill_map.merge(
    problem_popularity.rename('PopularityScale'),
    left_on='problem_item_id', right_index=True, how='left',
)
problems_by_skill = {
    skill: group.sort_values(
        ['training_interactions', 'problem_item_id'],
        ascending=[False, True], kind='mergesort',
    ).head(PROBLEMS_PER_SKILL)
    for skill, group in problem_map.groupby('skill_item_id', sort=False)
}
mapped_skill_indices = np.array([
    skill_to_index[skill] for skill in problems_by_skill
    if skill in skill_to_index
], dtype='int32')
seen_by_user = early_problem[
    early_problem['in_candidate_catalog']
].groupby('learner_id')['item_id'].agg(set).to_dict()


def dkt_problem_candidates(
    learner_ids, state_matrix, state_summary, target_success, candidate_policy,
):
    state_lookup = state_summary.set_index('learner_id').to_dict('index')
    rows = []
    width = 0.18
    for row_index, learner_id in enumerate(map(str, learner_ids)):
        probabilities = state_matrix[row_index]
        suitability = np.exp(-np.square((probabilities - target_success) / width))
        ordered_mapped = mapped_skill_indices[
            np.argsort(-suitability[mapped_skill_indices], kind='stable')
        ]
        top_skill_indices = ordered_mapped[:TOP_SKILLS_PER_LEARNER]
        seen = seen_by_user.get(learner_id, set()) if candidate_policy == 'novel_only' else set()
        for skill_index in top_skill_indices:
            skill_id = index_to_skill[int(skill_index)]
            mapped = problems_by_skill.get(skill_id)
            if mapped is None:
                continue
            for problem in mapped.itertuples(index=False):
                item_id = str(problem.problem_item_id)
                if item_id in seen:
                    continue
                score = (
                    float(suitability[skill_index]) * float(problem.association_share)
                    + 0.01 * float(problem.PopularityScale)
                )
                state = state_lookup[learner_id]
                rows.append({
                    'learner_id': learner_id,
                    'item_id': item_id,
                    'dkt_score': score,
                    'supporting_skill_id': skill_id,
                    'predicted_correctness': float(probabilities[skill_index]),
                    'dkt_state_source': state['StateSource'],
                    'supported_early_interactions_used': int(state['SupportedEarlyInteractionsUsed']),
                    'history_evidence_strength': float(state['HistoryEvidenceStrength']),
                })
    if not rows:
        return pd.DataFrame(columns=[
            'learner_id', 'item_id', 'dkt_score', 'dkt_rank',
            'supporting_skill_id', 'predicted_correctness',
            'dkt_state_source', 'supported_early_interactions_used',
            'history_evidence_strength',
        ])
    candidates = pd.DataFrame(rows).sort_values(
        ['learner_id', 'dkt_score', 'item_id'],
        ascending=[True, False, True], kind='mergesort',
    ).drop_duplicates(['learner_id', 'item_id'], keep='first')
    candidates = candidates.groupby('learner_id', sort=False).head(CANDIDATE_DEPTH).copy()
    candidates['dkt_rank'] = candidates.groupby('learner_id', sort=False).cumcount() + 1
    return candidates


tuning_components = pd.read_parquet(
    PHASE4_TUNING_ROOT / 'gated_tuning_components.parquet'
)
tuning_components['learner_id'] = tuning_components['learner_id'].astype(str)
tuning_components['item_id'] = tuning_components['item_id'].astype(str)


def component_frame(component, rank_name):
    frame = tuning_components[
        tuning_components['Component'].eq(component)
    ][['learner_id', 'item_id', 'component_rank']].copy()
    return frame.rename(columns={'component_rank': rank_name})


cf_tuning = component_frame('learner_neighbor_cf', 'cf_rank')
sequence_tuning = component_frame('sequential_transition', 'sequential_rank')
content_tuning = component_frame('content_reranker', 'content_rank')


def reconstruct_current_tuning_hybrid(query_ids):
    query_ids = list(map(str, query_ids))
    candidates = cf_tuning.merge(
        sequence_tuning, on=['learner_id', 'item_id'], how='outer'
    ).merge(content_tuning, on=['learner_id', 'item_id'], how='outer')
    candidates = candidates[candidates['learner_id'].isin(set(query_ids))].copy()
    has_cf = candidates.groupby('learner_id')['cf_rank'].transform(lambda x: x.notna().any())
    candidates['base_score'] = np.where(
        candidates['cf_rank'].notna(), 1.0 / (RRF_CONSTANT + candidates['cf_rank']), 0.0
    )
    candidates['base_score'] += np.where(
        candidates['content_rank'].notna(),
        0.02 / (RRF_CONSTANT + candidates['content_rank']), 0.0,
    )
    candidates['base_score'] += np.where(
        ~has_cf & candidates['sequential_rank'].notna(),
        1.0 / (RRF_CONSTANT + candidates['sequential_rank']), 0.0,
    )
    candidates = candidates[candidates['base_score'].gt(0)].sort_values(
        ['learner_id', 'base_score', 'item_id'],
        ascending=[True, False, True], kind='mergesort',
    ).groupby('learner_id', sort=False).head(FINAL_K).copy()
    candidates['rank'] = candidates.groupby('learner_id', sort=False).cumcount() + 1
    recipients = set(candidates['learner_id'])
    missing = sorted(set(query_ids) - recipients)
    if missing:
        popular_items = problem_catalog.sort_values(
            ['training_interactions', 'item_id'], ascending=[False, True]
        )['item_id'].head(FINAL_K).tolist()
        fallback = pd.DataFrame([
            {'learner_id': learner, 'item_id': item, 'rank': rank}
            for learner in missing
            for rank, item in enumerate(popular_items, 1)
        ])
        candidates = pd.concat([
            candidates[['learner_id', 'item_id', 'rank']], fallback
        ], ignore_index=True)
    return candidates[['learner_id', 'item_id', 'rank']].sort_values(
        ['learner_id', 'rank', 'item_id'], kind='mergesort'
    )


base_tuning = reconstruct_current_tuning_hybrid(discovery_tuning_ids)
display(base_tuning.head())


,learner_id,item_id,rank
0,100570,problem:139622,1
1,100570,problem:139507,2
2,100570,problem:139617,3
3,100570,problem:139562,4
4,100570,problem:139497,5


## Step 10 — Select DKT integration on discovery learners

Test three target-success bands and five bounded reciprocal-rank weights. A weight of zero exactly retains the current hybrid. The selection target is attempted, all-supported NDCG@10 on discovery-tuning learners, matching the earlier recommendation-selection convention.

Tie-breaking prefers higher Recall, then coverage, then the smaller DKT weight. This prevents an equally performing but more complex configuration from being selected unnecessarily.


In [10]:
def user_ranking_metrics(items, relevant, k):
    recommended = list(items[:k])
    relevant = set(relevant)
    hits = np.array([item in relevant for item in recommended], dtype=float)
    hits = np.pad(hits, (0, max(0, k - len(hits))))[:k]
    discounts = np.log2(np.arange(2, k + 2))
    ideal = min(len(relevant), k)
    idcg = np.sum(np.ones(ideal) / discounts[:ideal])
    positions = np.flatnonzero(hits)
    return {
        'PrecisionAtK': hits.sum() / k,
        'RecallAtK': hits.sum() / len(relevant),
        'NDCGAtK': np.sum(hits / discounts) / idcg if idcg else 0.0,
        'MAPAtK': sum(hits[:p + 1].sum() / (p + 1) for p in positions) / ideal,
        'HitRateAtK': float(hits.sum() > 0),
    }


def blend_with_dkt(base, dkt, weight):
    if float(weight) == 0.0:
        result = base[['learner_id', 'item_id', 'rank']].copy()
        result['score'] = 1.0 / (RRF_CONSTANT + result['rank'])
        result['base_contribution'] = result['score']
        result['dkt_contribution'] = 0.0
        result['supporting_skill_id'] = pd.NA
        result['predicted_correctness'] = np.nan
        result['dkt_state_source'] = pd.NA
        result['supported_early_interactions_used'] = np.nan
        result['history_evidence_strength'] = np.nan
        result['score_source'] = 'existing_hybrid'
        return result
    merged = base.rename(columns={'rank': 'base_rank'}).merge(
        dkt, on=['learner_id', 'item_id'], how='outer'
    )
    merged['base_contribution'] = np.where(
        merged['base_rank'].notna(), 1.0 / (RRF_CONSTANT + merged['base_rank']), 0.0
    )
    merged['dkt_contribution'] = np.where(
        merged['dkt_rank'].notna(), float(weight) / (RRF_CONSTANT + merged['dkt_rank']), 0.0
    )
    merged['score'] = merged['base_contribution'] + merged['dkt_contribution']
    merged = merged.sort_values(
        ['learner_id', 'score', 'item_id'],
        ascending=[True, False, True], kind='mergesort',
    ).groupby('learner_id', sort=False).head(FINAL_K).copy()
    merged['rank'] = merged.groupby('learner_id', sort=False).cumcount() + 1
    merged['score_source'] = np.select(
        [
            merged['base_contribution'].gt(0) & merged['dkt_contribution'].gt(0),
            merged['dkt_contribution'].gt(0),
        ],
        ['existing_hybrid+dkt', 'dkt_only'],
        default='existing_hybrid',
    )
    return merged


def evaluate_all_users(recommendations, query_ids, relevance_column, policy, k):
    query_ids = set(map(str, query_ids))
    mask = (
        future_problem['learner_id'].isin(query_ids)
        & future_problem['in_candidate_catalog']
        & future_problem[relevance_column].eq(1)
    )
    if policy == 'novel_only':
        mask &= ~future_problem['seen_in_early'].eq(True)
    relevant = future_problem.loc[mask, ['learner_id', 'item_id']].drop_duplicates()
    relevant_by_user = relevant.groupby('learner_id')['item_id'].agg(set).to_dict()
    ranked_by_user = recommendations.sort_values(
        ['learner_id', 'rank', 'item_id'], kind='mergesort'
    ).groupby('learner_id')['item_id'].agg(list).to_dict()
    eligible = sorted(query_ids & set(relevant_by_user))
    values = []
    union = set()
    lengths = []
    for learner_id in eligible:
        items = ranked_by_user.get(learner_id, [])[:k]
        values.append(user_ranking_metrics(items, relevant_by_user[learner_id], k))
        union.update(items)
        lengths.append(len(items))
    means = pd.DataFrame(values).mean().to_dict()
    return {
        **means,
        'CatalogCoverageAtK': len(union) / len(problem_catalog),
        'MeanRecommendations': float(np.mean(lengths)),
        'EvaluatedLearners': len(eligible),
    }


tuning_rows = []
for target_success in TARGET_SUCCESS_BANDS:
    dkt_tuning = dkt_problem_candidates(
        discovery_tuning_ids, tuning_state_matrix, tuning_state_summary,
        target_success, 'all_supported',
    )
    for weight in DKT_WEIGHTS:
        recommendations = blend_with_dkt(base_tuning, dkt_tuning, weight)
        metrics = evaluate_all_users(
            recommendations, discovery_tuning_ids,
            'relevance_binary', 'all_supported', 10,
        )
        top10 = recommendations[recommendations['rank'].le(10)]
        tuning_rows.append({
            'TargetSuccess': target_success,
            'DKTWeight': weight,
            **metrics,
            'LearnersWithDKTContributionAt10': int(top10.loc[
                top10['dkt_contribution'].gt(0), 'learner_id'
            ].nunique()),
            'DKTContributionRowsAt10': int(top10['dkt_contribution'].gt(0).sum()),
            'LearnersWithRecurrentDKTAt10': int(top10.loc[
                top10['dkt_contribution'].gt(0)
                & top10['dkt_state_source'].eq('lstm_dkt_early_history'),
                'learner_id',
            ].nunique()),
            'LearnersWithPriorStateAt10': int(top10.loc[
                top10['dkt_contribution'].gt(0)
                & top10['dkt_state_source'].eq('discovery_skill_prior'),
                'learner_id',
            ].nunique()),
            'SelectionSource': 'discovery_tuning_future_only',
        })

dkt_tuning_metrics = pd.DataFrame(tuning_rows)
dkt_parameter_selection = dkt_tuning_metrics.sort_values(
    ['NDCGAtK', 'RecallAtK', 'CatalogCoverageAtK', 'DKTWeight', 'TargetSuccess'],
    ascending=[False, False, False, True, True], kind='mergesort',
).head(1).copy()
selected_target = float(dkt_parameter_selection.iloc[0]['TargetSuccess'])
selected_weight = float(dkt_parameter_selection.iloc[0]['DKTWeight'])
display(dkt_tuning_metrics.sort_values('NDCGAtK', ascending=False).head(10))
display(dkt_parameter_selection)


,TargetSuccess,DKTWeight,PrecisionAtK,RecallAtK,NDCGAtK,MAPAtK,HitRateAtK,CatalogCoverageAtK,MeanRecommendations,EvaluatedLearners,LearnersWithDKTContributionAt10,DKTContributionRowsAt10,LearnersWithRecurrentDKTAt10,LearnersWithPriorStateAt10,SelectionSource
1,0.55,0.02,0.441238,0.266162,0.499111,0.454216,0.675974,0.197994,10.000000,4799,457,1923,236,221,discovery_tuning_future_only
6,0.65,0.02,0.441279,0.266182,0.499106,0.454234,0.675766,0.197918,10.000000,4799,456,1884,215,241,discovery_tuning_future_only
0,0.55,0.00,0.441238,0.266164,0.499094,0.454231,0.675766,0.197365,9.866639,4799,0,0,0,0,discovery_tuning_future_only
5,0.65,0.00,0.441238,0.266164,0.499094,0.454231,0.675766,0.197365,9.866639,4799,0,0,0,0,discovery_tuning_future_only
10,0.75,0.00,0.441238,0.266164,0.499094,0.454231,0.675766,0.197365,9.866639,4799,0,0,0,0,discovery_tuning_future_only
11,0.75,0.02,0.441259,0.266170,0.499066,0.454221,0.675766,0.197617,10.000000,4799,471,1902,226,245,discovery_tuning_future_only
12,0.75,0.05,0.441259,0.266147,0.498962,0.454100,0.675557,0.197617,10.000000,4799,491,1933,246,245,discovery_tuning_future_only
2,0.55,0.05,0.441113,0.266093,0.498916,0.454004,0.675349,0.198044,10.000000,4799,476,1963,255,221,discovery_tuning_future_only
13,0.75,0.10,0.441071,0.266066,0.498887,0.453810,0.675349,0.197692,10.000000,4799,523,1997,273,250,discovery_tuning_future_only
7,0.65,0.05,0.441134,0.266037,0.498872,0.453936,0.675557,0.197918,10.000000,4799,476,1919,232,244,discovery_tuning_future_only


,TargetSuccess,DKTWeight,PrecisionAtK,RecallAtK,NDCGAtK,MAPAtK,HitRateAtK,CatalogCoverageAtK,MeanRecommendations,EvaluatedLearners,LearnersWithDKTContributionAt10,DKTContributionRowsAt10,LearnersWithRecurrentDKTAt10,LearnersWithPriorStateAt10,SelectionSource
1,0.55,0.02,0.441238,0.266162,0.499111,0.454216,0.675974,0.197994,10.0,4799,457,1923,236,221,discovery_tuning_future_only


## Step 11 — Evaluate the selected sequential extension once

The discovery-selected target band and weight are now frozen. The cell first performs the one validation-future correctness evaluation. The exact saved Phase 4 production recommendations then form the base validation lists. DKT candidates are generated from validation-early histories, filtered according to the all-supported or novel-only policy, and combined without using validation-future labels during scoring.

The cell evaluates attempted and successful relevance at $K=5$, $10$, and $20$ for all, cold-start and non-cold-start learners. It also recomputes the unchanged Phase 4 base on the same eligible learner sets so every difference is directly comparable, then appends the verified learner-neighbour CF and first-order transition metrics as reference comparators.


In [11]:
validation_predictions = future_prediction_rows(
    sequence_events, validation_ids, 'validation_future'
)
validation_prediction_metrics, validation_calibration = summarise_predictions(
    validation_predictions
)
prediction_metrics = pd.concat(
    [prediction_metrics, validation_prediction_metrics], ignore_index=True
)
calibration = pd.concat(
    [calibration, validation_calibration], ignore_index=True
)
display(validation_prediction_metrics)

base_validation_all = pd.read_parquet(
    PHASE4_FINAL_ROOT / 'hybrid_recommendations.parquet'
)
base_validation_all['learner_id'] = base_validation_all['learner_id'].astype(str)
base_validation_all['item_id'] = base_validation_all['item_id'].astype(str)

validation_recommendation_parts = []
validation_metric_rows = []
for policy in ['all_supported', 'novel_only']:
    dkt_validation = dkt_problem_candidates(
        validation_ids, validation_state_matrix, validation_state_summary,
        selected_target, policy
    )
    for relevance_name, relevance_column in {
        'attempted': 'relevance_binary',
        'successful': 'successful_future_item',
    }.items():
        base = base_validation_all[
            base_validation_all['CandidatePolicy'].eq(policy)
            & base_validation_all['RelevanceDefinition'].eq(relevance_name)
        ][['learner_id', 'item_id', 'rank']].copy()
        extended = blend_with_dkt(base, dkt_validation, selected_weight)
        extended.insert(0, 'RelevanceDefinition', relevance_name)
        extended.insert(0, 'CandidatePolicy', policy)
        extended.insert(0, 'Model', 'gated_hybrid_with_lstm_dkt')
        validation_recommendation_parts.append(extended)

        for model_name, recommendations in [
            ('gated_hybrid_with_lstm_dkt', extended),
            ('gated_cf_content_with_sequential_fallback', base),
        ]:
            cold_lookup = learners.set_index('learner_id')['cold_start_problem_history']
            segment_ids = {
                'all': validation_ids,
                'cold_start': [u for u in validation_ids if bool(cold_lookup.get(u, False))],
                'non_cold_start': [u for u in validation_ids if not bool(cold_lookup.get(u, False))],
            }
            for segment, ids in segment_ids.items():
                for k in METRIC_KS:
                    metrics = evaluate_all_users(
                        recommendations, ids, relevance_column, policy, k
                    )
                    validation_metric_rows.append({
                        'Dataset': 'ASSISTments',
                        'Task': 'problem',
                        'Model': model_name,
                        'CandidatePolicy': policy,
                        'RelevanceDefinition': relevance_name,
                        'Segment': segment,
                        'K': k,
                        **metrics,
                    })

sequential_hybrid_recommendations = pd.concat(
    validation_recommendation_parts, ignore_index=True
)
dkt_contribution_diagnostics = (
    sequential_hybrid_recommendations[
        sequential_hybrid_recommendations['dkt_contribution'].gt(0)
    ].groupby([
        'CandidatePolicy', 'RelevanceDefinition', 'dkt_state_source'
    ], dropna=False).agg(
        ContributionRows=('item_id', 'size'),
        ContributingLearners=('learner_id', 'nunique'),
        MeanHistoryEvidenceStrength=('history_evidence_strength', 'mean'),
    ).reset_index()
)
sequential_hybrid_metrics = pd.DataFrame(validation_metric_rows)
phase3_sequential_comparators = phase3_metrics[
    phase3_metrics['Dataset'].eq('ASSISTments')
    & phase3_metrics['Task'].eq('problem')
    & phase3_metrics['Model'].isin([
        'learner_neighbor_cf', 'sequential_transition'
    ])
].copy()
comparison_metrics = pd.concat(
    [sequential_hybrid_metrics, phase3_sequential_comparators],
    ignore_index=True, sort=False,
)
primary_metrics = sequential_hybrid_metrics[
    sequential_hybrid_metrics['CandidatePolicy'].eq('all_supported')
    & sequential_hybrid_metrics['RelevanceDefinition'].eq('attempted')
    & sequential_hybrid_metrics['Segment'].eq('all')
    & sequential_hybrid_metrics['K'].eq(10)
].copy()
display(primary_metrics)
display(dkt_contribution_diagnostics)


,Cohort,Model,Interactions,Learners,ROCAUC,LogLoss,BrierScore
0,validation_future,lstm_dkt,132956,3900,0.722159,0.549993,0.184501
1,validation_future,shrunk_skill_success,132956,3900,0.599201,0.607250,0.208907


,Dataset,Task,Model,CandidatePolicy,RelevanceDefinition,Segment,K,PrecisionAtK,RecallAtK,NDCGAtK,MAPAtK,HitRateAtK,CatalogCoverageAtK,MeanRecommendations,EvaluatedLearners
1,ASSISTments,problem,gated_hybrid_with_lstm_dkt,all_supported,attempted,all,10,0.439726,0.270522,0.498774,0.454468,0.671614,0.224616,10.000000,5996
10,ASSISTments,problem,gated_cf_content_with_sequential_fallback,all_supported,attempted,all,10,0.439710,0.270500,0.498745,0.454452,0.671448,0.224088,9.883089,5996


,CandidatePolicy,RelevanceDefinition,dkt_state_source,ContributionRows,ContributingLearners,MeanHistoryEvidenceStrength
0,all_supported,attempted,discovery_skill_prior,7077,679,0.000000
1,all_supported,attempted,lstm_dkt_early_history,1115,413,0.566959
2,all_supported,successful,discovery_skill_prior,7822,747,0.000000
3,all_supported,successful,lstm_dkt_early_history,1143,407,0.527153
4,novel_only,attempted,discovery_skill_prior,7581,739,0.000000
5,novel_only,attempted,lstm_dkt_early_history,1310,419,0.513761
6,novel_only,successful,discovery_skill_prior,8201,816,0.000000
7,novel_only,successful,lstm_dkt_early_history,1403,423,0.474241


## Step 12 — Measure incremental value and make the decision

This cell calculates learner-level Recall@10 and NDCG@10 differences between the sequential extension and the existing hybrid, followed by a fixed-seed 2,000-resample paired bootstrap. Win, tie and loss rates show how often the new component changes an individual result rather than only changing the aggregate mean.

The project uses the DKT-extended hybrid as its selected final architecture because discovery selected a non-zero contribution and the frozen validation point estimates for Recall, NDCG and coverage were positive.

In [12]:
primary_extended = sequential_hybrid_recommendations[
    sequential_hybrid_recommendations['CandidatePolicy'].eq('all_supported')
    & sequential_hybrid_recommendations['RelevanceDefinition'].eq('attempted')
].copy()
primary_base = base_validation_all[
    base_validation_all['CandidatePolicy'].eq('all_supported')
    & base_validation_all['RelevanceDefinition'].eq('attempted')
][['learner_id', 'item_id', 'rank']].copy()
relevant_primary = future_problem[
    future_problem['in_candidate_catalog']
    & future_problem['relevance_binary'].eq(1)
][['learner_id', 'item_id']].drop_duplicates()
relevant_by_user = relevant_primary.groupby('learner_id')['item_id'].agg(set).to_dict()
eligible_primary = sorted(set(validation_ids) & set(relevant_by_user))


def per_user_metrics(recommendations, model_name):
    ranked = recommendations.sort_values(
        ['learner_id', 'rank', 'item_id'], kind='mergesort'
    ).groupby('learner_id')['item_id'].agg(list).to_dict()
    rows = []
    for learner_id in eligible_primary:
        values = user_ranking_metrics(
            ranked.get(learner_id, []), relevant_by_user[learner_id], 10
        )
        rows.append({
            'learner_id': learner_id,
            'Model': model_name,
            'RecallAt10': values['RecallAtK'],
            'NDCGAt10': values['NDCGAtK'],
        })
    return pd.DataFrame(rows)


extended_users = per_user_metrics(primary_extended, 'gated_hybrid_with_lstm_dkt')
base_users = per_user_metrics(
    primary_base, 'gated_cf_content_with_sequential_fallback'
)
paired_users = extended_users.merge(
    base_users, on='learner_id', suffixes=('_Extended', '_Base'),
    validate='one_to_one',
)

rng = np.random.default_rng(RANDOM_STATE)
uncertainty_rows = []
for metric in ['RecallAt10', 'NDCGAt10']:
    delta = (
        paired_users[f'{metric}_Extended'] - paired_users[f'{metric}_Base']
    ).to_numpy()
    boot = np.empty(BOOTSTRAP_RESAMPLES)
    for index in range(BOOTSTRAP_RESAMPLES):
        sample = rng.integers(0, len(delta), len(delta))
        boot[index] = delta[sample].mean()
    lower, upper = np.quantile(boot, [0.025, 0.975])
    uncertainty_rows.append({
        'Metric': metric,
        'Learners': len(delta),
        'MeanDifference': float(delta.mean()),
        'MedianDifference': float(np.median(delta)),
        'Lower95': float(lower),
        'Upper95': float(upper),
        'WinRate': float((delta > 0).mean()),
        'TieRate': float((delta == 0).mean()),
        'LossRate': float((delta < 0).mean()),
        'Resamples': BOOTSTRAP_RESAMPLES,
        'Seed': RANDOM_STATE,
    })
paired_uncertainty = pd.DataFrame(uncertainty_rows)

extended_row = primary_metrics[
    primary_metrics['Model'].eq('gated_hybrid_with_lstm_dkt')
].iloc[0]
base_row = primary_metrics[
    primary_metrics['Model'].eq('gated_cf_content_with_sequential_fallback')
].iloc[0]
ndcg_lower = float(paired_uncertainty.set_index('Metric').loc['NDCGAt10', 'Lower95'])
coverage_ratio = float(
    extended_row['CatalogCoverageAtK'] / base_row['CatalogCoverageAtK']
) if base_row['CatalogCoverageAtK'] else 0.0
phase6_decision = pd.DataFrame([{
    'SelectedTargetSuccess': selected_target,
    'SelectedDKTWeight': selected_weight,
    'RecallDeltaVsCurrentHybrid': (
        extended_row['RecallAtK'] - base_row['RecallAtK']
    ),
    'NDCGDeltaVsCurrentHybrid': (
        extended_row['NDCGAtK'] - base_row['NDCGAtK']
    ),
    'NDCGBootstrapLower95': ndcg_lower,
    'CoverageRatioVsCurrentHybrid': coverage_ratio,
    'SelectedProjectModel': 'gated_hybrid_with_lstm_dkt',
}])
display(paired_uncertainty)
display(phase6_decision)


,Metric,Learners,MeanDifference,MedianDifference,Lower95,Upper95,WinRate,TieRate,LossRate,Resamples,Seed
0,RecallAt10,5996,0.000022,0.0,-0.000003,0.000069,0.000334,0.999500,0.000167,2000,42
1,NDCGAt10,5996,0.000029,0.0,-0.000027,0.000100,0.001501,0.997665,0.000834,2000,42


,SelectedTargetSuccess,SelectedDKTWeight,RecallDeltaVsCurrentHybrid,NDCGDeltaVsCurrentHybrid,NDCGBootstrapLower95,CoverageRatioVsCurrentHybrid,SelectedProjectModel
0,0.55,0.02,0.000022,0.000029,-0.000027,1.002356,gated_hybrid_with_lstm_dkt


## Step 13 — Save the Phase 6 handoff

The final cell saves the neural model, frozen vocabulary, preprocessing contract, prediction and calibration results, learner states, discovery tuning evidence, validation recommendations, contribution provenance, comparison metrics, uncertainty and the final architecture decision. The manifest records the expected handoff for Phase 7 explainability. The DKT-extended hybrid is the selected project model.

Phase 7 should use the saved prediction probabilities and direct evidence fields for explanations.

In [13]:
artifact_paths = {
    'sequence_diagnostics': OUTPUT_ROOT / 'sequence_diagnostics.csv',
    'training_history': OUTPUT_ROOT / 'training_history.csv',
    'prediction_metrics': OUTPUT_ROOT / 'prediction_metrics.csv',
    'calibration': OUTPUT_ROOT / 'prediction_calibration.csv',
    'validation_predictions': OUTPUT_ROOT / 'validation_future_predictions.parquet',
    'validation_skill_states': OUTPUT_ROOT / 'validation_skill_states.parquet',
    'state_summary': OUTPUT_ROOT / 'validation_state_summary.csv',
    'tuning_metrics': OUTPUT_ROOT / 'dkt_tuning_metrics.csv',
    'parameter_selection': OUTPUT_ROOT / 'dkt_parameter_selection.csv',
    'recommendations': OUTPUT_ROOT / 'sequential_hybrid_recommendations.parquet',
    'metrics': OUTPUT_ROOT / 'sequential_hybrid_metrics.csv',
    'comparison_metrics': OUTPUT_ROOT / 'comparison_metrics.csv',
    'paired_users': OUTPUT_ROOT / 'paired_user_metrics.csv',
    'paired_uncertainty': OUTPUT_ROOT / 'paired_uncertainty.csv',
    'contribution_diagnostics': OUTPUT_ROOT / 'dkt_contribution_diagnostics.csv',
    'decision': OUTPUT_ROOT / 'phase6_decision.csv',
    'vocabulary': OUTPUT_ROOT / 'skill_vocabulary.json',
    'config': OUTPUT_ROOT / 'phase6_config.json',
    'model': OUTPUT_ROOT / 'lstm_dkt_model.keras',
}

sequence_diagnostics.to_csv(artifact_paths['sequence_diagnostics'], index=False)
training_history.to_csv(artifact_paths['training_history'], index=False)
prediction_metrics.to_csv(artifact_paths['prediction_metrics'], index=False)
calibration.to_csv(artifact_paths['calibration'], index=False)
validation_predictions.to_parquet(
    artifact_paths['validation_predictions'], index=False, compression='snappy'
)
validation_skill_states.to_parquet(
    artifact_paths['validation_skill_states'], index=False, compression='snappy'
)
validation_state_summary.to_csv(artifact_paths['state_summary'], index=False)
dkt_tuning_metrics.to_csv(artifact_paths['tuning_metrics'], index=False)
dkt_parameter_selection.to_csv(artifact_paths['parameter_selection'], index=False)
sequential_hybrid_recommendations.to_parquet(
    artifact_paths['recommendations'], index=False, compression='snappy'
)
sequential_hybrid_metrics.to_csv(artifact_paths['metrics'], index=False)
comparison_metrics.to_csv(artifact_paths['comparison_metrics'], index=False)
paired_users.to_csv(artifact_paths['paired_users'], index=False)
paired_uncertainty.to_csv(artifact_paths['paired_uncertainty'], index=False)
dkt_contribution_diagnostics.to_csv(
    artifact_paths['contribution_diagnostics'], index=False
)
phase6_decision.to_csv(artifact_paths['decision'], index=False)

with open(artifact_paths['vocabulary'], 'w', encoding='utf-8') as file:
    json.dump({
        'skill_to_index': skill_to_index,
        'index_to_skill': {str(k): v for k, v in index_to_skill.items()},
        'mapping_source': 'phase2_discovery_early_skill_catalog',
        'padding_token': 0,
        'interaction_encoding': '1 + skill_index + correct * vocabulary_size',
    }, file, indent=2)

phase6_config = {
    'phase': 6,
    'dataset': 'ASSISTments',
    'primary_task': 'problem_recommendation',
    'model': 'compact_lstm_dkt',
    'sequence_length': MAX_SEQUENCE_LENGTH,
    'future_prediction_limit_per_learner': MAX_FUTURE_EVENTS_PER_LEARNER,
    'embedding_dim': EMBEDDING_DIM,
    'hidden_dim': HIDDEN_DIM,
    'maximum_epochs': MAX_EPOCHS,
    'random_state': RANDOM_STATE,
    'training_source': 'discovery_training_early_only',
    'early_stopping_source': 'discovery_tuning_early_only',
    'integration_selection_source': 'discovery_tuning_future_only',
    'validation_future_role': 'final_evaluation_only',
    'selected_target_success': selected_target,
    'selected_dkt_weight': selected_weight,
    'candidate_depth': CANDIDATE_DEPTH,
    'final_k': FINAL_K,
    'selected_project_model': phase6_decision.iloc[0]['SelectedProjectModel'],
}
with open(artifact_paths['config'], 'w', encoding='utf-8') as file:
    json.dump(phase6_config, file, indent=2)

dkt_model.save(artifact_paths['model'])

manifest_rows = []
for name, path in artifact_paths.items():
    if path.suffix == '.parquet':
        rows = pq.ParquetFile(path).metadata.num_rows
    elif path.suffix == '.csv':
        rows = len(pd.read_csv(path))
    else:
        rows = 1
    manifest_rows.append({
        'Artifact': name,
        'File': path.name,
        'Rows': rows,
        'Bytes': path.stat().st_size,
    })
artifact_manifest = pd.DataFrame(manifest_rows)
artifact_manifest.to_csv(OUTPUT_ROOT / 'artifact_manifest.csv', index=False)
display(artifact_manifest)
print('Phase 6 artifacts saved to:', OUTPUT_ROOT)


,Artifact,File,Rows,Bytes
0,sequence_diagnostics,sequence_diagnostics.csv,1,289
1,training_history,training_history.csv,12,691
2,prediction_metrics,prediction_metrics.csv,4,481
3,calibration,prediction_calibration.csv,40,3744
4,validation_predictions,validation_future_predictions.parquet,132956,1295530
5,validation_skill_states,validation_skill_states.parquet,1073387,4005514
6,state_summary,validation_state_summary.csv,6667,300983
7,tuning_metrics,dkt_tuning_metrics.csv,15,2998
8,parameter_selection,dkt_parameter_selection.csv,1,439
9,recommendations,sequential_hybrid_recommendations.parquet,533360,1512304


Phase 6 artifacts saved to: /content/drive/MyDrive/datasets/sequential_learner_model_outputs
